# **A vetorização _Word-Embedding_**
<font size=3>

*Word-embeddings* representam palavras/tokens como **vetores densos** compostos por números racionais, permitindo representações com **dimensões significativamente menores** em comparação com a codificação *one-hot*, BoW ou TF-IDF. Esses vetores são **aprendidos durante o treinamento** ou derivados de embeddings pré-treinados, que são especialmente úteis para conjuntos de dados menores.

Por exemplo, uma palavra representada em codificação *one-hot* como [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] pode ser uma forma *embedding* ("embutida") como [0.245 -0.183 0.834]. O tamanho de um vetor *one-hot* corresponde ao **tamanho do vocabulário** ($\mathtt{vocab\_size}$), enquanto o tamanho de um vetor *embedding* é determinado pela **dimensão do _embedding_** ($\mathtt{embed\_dim}$), um hiperparâmetro pré-definido.



## **1. Como essa vetorização funciona na prática:**
<font size=3>

1. Importamos o *corpus*;

2. Transformamos cada sentença em uma lista de $\mathtt{token\_IDs}$:
    - Depois que todos os *tokens* são catalogados na lista de vocabulário, cada *token* de uma sentença é substituído pelo índice de sua posição da lista de vocabulário.
    <br>

3. Realizamos o *word-embedding*:
    - Inicializamos randomicamente a **matriz _embedding_**, $\,W_e \in \mathbb{R}^{\mathtt{vocab\_size} \times \mathtt{emded\_dim}}\,$;
    - Cada linha de $W_e$ irá corresponder a um ***token* vetorizado**.
    - Durante o treinamento da rede neural, *weights*, *bias* e os elementos da matriz *embedding* são ajustados **de modo a extrair o máximo de _features_ possíveis** das sentenças.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/')

In [1]:
import numpy as np
from keras import layers

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 1. importando o corpus:
corpus = ["Beautiful is better than ugly",
          "Explicit is better than implicit",
          "Simple is better than complex",
          "Complex is better than complicated",
          "Flat is better than nested",
          "Sparse is better than dense",
          "Readability counts",
          "Special cases aren't special enough to break the rules",
          "Although practicality beats purity",
          "Errors should never pass silently",
          "Unless explicitly silenced",
          "In the face of ambiguity, refuse the temptation to guess",
          "There should be one -- and preferably only one -- obvious way to do it",
          "Although that way may not be obvious at first unless you're Dutch",
          "Now is better than never",
          "Although never is often better than right now",
          "If the implementation is hard to explain, it's a bad idea",
          "If the implementation is easy to explain, it may be a good idea",
          "Namespaces are one honking great idea -- let's do more of those!"]


In [ ]:
# 2. transformando cada sentença em uma lista de token-IDs:

vocab_size = None # captura o tamanho máximo do vocabulário
max_len = 7 # tamanho máximo da sentença

vectorizer = layers.TextVectorization(max_tokens=vocab_size,
                                     standardize='lower_and_strip_punctuation',
                                     split='whitespace',
                                     output_mode='int',
                                     output_sequence_length=max_len)

vectorizer.adapt(corpus)

vocab = vectorizer.get_vocabulary()
vocab_size = vectorizer.vocabulary_size()

# obtendo as listas de tokens-IDs:
token_ids = vectorizer(corpus)

# [UNK] = unknown word
print("Tamanho do vocabulário:", vocab_size)
print("Tokens do vocabulário:", vocab)

In [ ]:
print("Zen of Python indexado:")
token_ids

In [ ]:
# 3. realizando o word-embedding:
embed_dim = 3

embedding = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)


# Para que possamos visualizar a matriz embedding W_e, precisamos inicializar
# o objeto `embedding` com o formato da entrada (sentenças vetorizadads):
embedding.build(input_shape=(token_ids.shape))


# imprimindo a matriz embedding W_e - uma matriz do formato (vocab_size, embed_dim):
W_e = embedding.weights[0].numpy()

print("Matrix embedding (W_e):", W_e.shape, "\n")
print(W_e)

In [ ]:
# obtendo os tokens vetorizados:
embed_tokens = embedding(token_ids)

In [ ]:
# vamos visualizar uma sentença vetorizada:

i = 0
print(f"- Sentença:\n{corpus[i]}\n")
print(f"- Token-IDs:\n{token_ids[i]}\n")
print(f"- Embedded tokens:\n{embed_tokens[0].numpy()}")

<font size=3>

> Observe acima que o $\mathtt{token\_IDs}$ **são** os índices das linhas da matriz embedding $W_e$. Assim, cada *token* é mapeada por um vetor de tamanho $\mathtt{embed\_dim}$.

## **2. _Word-embeddings_ pré-treinadas:**
<font size=3>

Ao trabalhar com um **pequeno conjunto de dados** ou com o objetivo de reduzir os custos computacionais de treinamento, **_word-embeddings_ pré-treinados** podem ser altamente benéficos. [*Word-embeddings* pré-treinados](https://keras.io/examples/nlp/pretrained_word_embeddings/) populares incluem [Word2vec](https://code.google.com/archive/p/word2vec/) e [Global Vectors for Word Representation (GloVe)](https://nlp.stanford.edu/projects/glove/). Neste exemplo, vetorizaremos nosso corpus *Zen of Python* usando *embeddings* do **GloVe**.

Para começar, precisamos baixar o conjunto de dados GloVe usando a célula abaixo. O arquivo zip baixado contém embeddings com quatro representações dimensionais diferentes (50D, 100D, 200D e 300D). Para esta tarefa, nos concentraremos em $\mathtt{embed\_dim = 100}$.

In [ ]:
#!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
#!unzip -q glove.6B.zip

In [ ]:
# obtendo os vetores de tokens:
embed_dim = 100
embed_dict = {}

with open(f"/content/drive/MyDrive/IMD1107 - Processamento de Linguagem Natural/3-unidade/dataset/glove.6B.{embed_dim}d.txt") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embed_dict[word] = coefs

print(f"Temos {len(embed_dict)} palavras vetorizadas.\n")

print('Vetorização da palavra "talk":\n', embed_dict["talk"])

<font size=3>

Não precisamos considerar todos os pesos do *embedding* do GloVe, que têm o formato (400000, 100). Em vez disso, se quisermos resolver a tarefa usando apenas o corpus _Zen of Python_, podemos simplesmente nos concentrar em seu vocabulário. Isso nos permitirá obter um array com o formato (82, 100).

In [ ]:
hits = 0
misses = 0

# definindo um dicionário de vocabulário do corpus:
vocab_dict = dict(zip(vocab, range(len(vocab))))

# preparando os pesos (weights) da matriz embedding:
embedding_weights = np.zeros((vocab_size, embed_dim))

for word, i in vocab_dict.items():
    embedding_vector = embed_dict.get(word)
    if embedding_vector is not None:

        '''
        Palavras que não estão nos índices do embedding serão definidas como zero.
        Isso também se aplica a representação de preenchimento (padding) e "out of vocabulary" (OOV).
        '''

        embedding_weights[i] = embedding_vector
        hits += 1

    else:
        misses += 1

print(f"Converted {hits} words ({misses} misses)")

In [ ]:
# definindo o word-embeding pré-treinado:

embedding = layers.Embedding(input_dim=vocab_size,
                             output_dim=embed_dim,
                             weights=[embedding_weights],
                             trainable=False)

'''
Como estamos usando weights pré-treinados da matriz embedding, não queremos
perdê-los durante o treinamento da NN. Portanto, para os embedding weights,
nós definimos `trainable=False`.
'''

print(f"weights array:{embedding.weights[0].shape}")

# get the embedded tokens:
embed_tokens = embedding(token_ids)

In [ ]:
# vamos visualizar uma sentença vetorizada:

i = 0
print(f"- Sentença:\n{corpus[i]}\n")
print(f"- Token-IDs:\n{token_ids[i]}\n")
print(f"- Embedded tokens:\n{embed_tokens[0].numpy()}")

<font size=3>
<br>
    
Uma vantagem fundamental de *word-embeddings* é sua capacidade de capturar as relações semânticas entre as palavras no **espaço de _embedding_**. Por exemplo, sinônimos tendem a ter representações vetoriais semelhantes, resultando em distâncias geométricas mais próximas. Isso permite relações intuitivas, como "cachorro" estar mais próximo de "lobo", e "gato" de "tigre".


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# selecionando palabras da matriz de embedding do GloVe:
word_vectors = []
words = ['man', 'woman', 'king', 'queen']

for word in words:
    if word in embed_dict:
        word_vectors.append(embed_dict[word])
    else:
        print(f"Palavra '{word}' não encontrada no GloVe!")

word_vectors = np.array(word_vectors)

word_vectors.shape

In [ ]:
# redução de dimensionalidade para plote do espaço vetorial:
pca = PCA(n_components=2)

vectors_2D = pca.fit_transform(word_vectors)

vectors_2D.shape

In [ ]:
for word, vec in zip(words, vectors_2D):
    plt.scatter(*vec, alpha=0.6)
    plt.text(*vec+0.05, word)

plt.axhline(c="tab:blue", lw=2, alpha=0.5)
plt.axvline(c="tab:orange", lw=2, alpha=0.5)
plt.xlim(-4, 4)
plt.ylim(-3, 3)
plt.xlabel("$PC_1$")
plt.ylabel("$PC_2$")
plt.grid()
plt.show()

<font size=3>

Essa "distância geométrica" é tipicamente medida pela similaridade de cosseno,
$$
    \cos\theta = \frac{\vec a\cdot \vec b}{|\vec a||\vec b|} \, ,
$$
onde $\theta \in [-1,\, +1]$ varia de vetores opostos (-1) a semelhantes (+1).

O conjunto de dados GloVe inclui palavras de vários idiomas, o que significa que algumas correlações opostas podem surgir entre palavras de diferentes idiomas. Abaixo, alguns exemplos de similaridade de cosseno usando apenas palavras em inglês.

In [ ]:
def cosine_similarity(a, b):

    A = np.linalg.norm(a)
    B = np.linalg.norm(b)

    return np.dot(a, b)/(A*B)

print("(good vs nice):", cosine_similarity(embed_dict["good"], embed_dict["nice"]))
print("(dog vs universities):", cosine_similarity(embed_dict["dog"], embed_dict["universities"]))
print("(read vs colonizer):", cosine_similarity(embed_dict["read"], embed_dict["colonizer"]))

<font size=3>
<br>
    
Os resultados da similaridade de cosseno ilustram perfeitamente como os *embeddings* capturam o significado contextual da linguagem. O valor positivo alto ($0.73$) entre **"_good_"** e **"_nice_"** demonstra que o modelo aprendeu a forte relação de sinonímia entre elas, posicionando seus vetores em direções muito próximas. Em nítido contraste, o valor quase zero ($0.002$) para **"_dog_"** e **"_universities_"** mostra que o modelo os considera semanticamente irrelevantes (ortogonais), pois não compartilham contexto. O mais sutil é o valor negativo ($-0.29$) entre **"_read_"** e **"_colonizer_"**, que indica não apenas uma falta de relação, mas um contraste contextual.

## **3. [Extra] Camada linear _vs_ camada _embedding_:**
<font size=3>

Diz-se que as **camadas de _embeddings_** funcionam de forma semelhante às **camadas lineares**. Para entender melhor como funciona a incorporação de palavras, vamos comparar suas semelhanças e diferenças.

<font size=3>

Uma camada *embedding* se comporta de forma semelhante a uma camada linear, pois não possui uma função de ativação (já que é linear), mas não inclui um vetor *bias*. Ao considerar a entrada como um **vetor _one-hot_** $a_0^i$, a saída da camada linear (para $b_1^i = 0$) é dado por,
\begin{align}
     a_0^i\;\; W^{ij} &= a_1^j \, ,\\\\
     \begin{pmatrix}
     0 & 1 & 0 & 0
     \end{pmatrix}
     \begin{pmatrix}
         w_{00} & w_{01} & w_{02} \\
         w_{10} & w_{11} & w_{12} \\
         w_{20} & w_{21} & w_{22} \\
         w_{30} & w_{31} & w_{32}
     \end{pmatrix}
     &=
     \begin{pmatrix}
         w_{10} & w_{11} & w_{12}
     \end{pmatrix}  \, ,
\end{align}
onde os índices $\; i \in [0,\, \mathtt{vocab\_size})$ e $\; j \in [0,\, \mathtt{embed\_dim})$. Já, a camada *embedding* funciona da seguinte forma,
\begin{align}
    \delta^{ki}\;\;W^{ij} &= a_1^j\, ,\\\\
     \begin{pmatrix}
         w_{00} & w_{01} & w_{02} \\
         [w_{10} & w_{11} & w_{12}] \\
         w_{20} & w_{21} & w_{22} \\
         w_{30} & w_{31} & w_{32}
     \end{pmatrix}
      &\Rightarrow
     \begin{pmatrix}
         w_{10} & w_{11} & w_{12}
     \end{pmatrix} \, ,
\end{align}
onde $\delta^{kj} = [1,\; k=j;\quad 0,\; k\neq j]$, e $k = 1$ é um $\mathtt{token\_id}$, que seleciona o vetor $(w_{10}\;\; w_{11}\;\;  w_{12})$ sem precisar de uma codificação *one-hot*.

### **3.1 Camada linear:**

In [ ]:
# vamos considerar um corpus com vocabulário de tamanho:
vocab_size = 10

# e que iremos fazer o embedding em uma matriz de dimensão:
embed_dim = 5

# também iremos considerar a seguinte lista de token-IDs:
token_ids = np.array([3, 8, 2, 5, 0])

'''
Note que ID = 0 em tokens-IDs representa o preenchimento (padding).
Para comparar camada linear vs embedding, o tamnho da entrada da camada (max-len)
precisa ter o mesmo tamanho de embed_dim.
'''

max_len = token_ids.size

assert max_len == embed_dim

In [ ]:
# vetorização one-hot do token-IDs:
onehot = layers.CategoryEncoding(num_tokens=vocab_size, output_mode="one_hot")(token_ids)

for ID, hot in zip(token_ids, onehot):
    print(f"{ID}: {hot}")


In [ ]:
# definindo a camada linear:
linear = layers.Dense(units=max_len, bias_initializer="zeros")
linear.build(input_shape=onehot.shape) # incializando os weights

weights = linear.weights[0].numpy()

print("- Weights lineares:\n", weights, end="\n\n")
print("- Saída da camada linear:\n", linear(onehot).numpy())

### **3.2 Camada _embedding_:**
<font size=3>

Aqui, a camada *embedding* seleciona as linhas de $\mathtt{weights}$ usando $\mathtt{token\_ids}$, sem a necessidade de multiplicação de matriz da codificação *one-hot*.

In [ ]:
# definindo a camada embedding usando os mesmos weights da camada linear, para comparação:
embedding = layers.Embedding(input_dim=vocab_size, output_dim=max_len, weights=[weights])

print("- Weights do embedding:\n", embedding.weights[0].numpy(), end="\n\n")
print("- Saída da camada de embedding:\n", embedding(token_ids).numpy())

<font size=3>
<br>
    
Assim, mostramos os **mesmos resultados** de saída de ambos os tipos de camadas!

## __Referências:__
<font size=3>
    
 - [Deep Learning with Python](https://books.google.com.br/books/about/Deep_Learning_with_Python.html?id=Yo3CAQAACAAJ&redir_esc=y);
 - [Build a Large Language Model From Scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb);
 - [Understanding word-embedding with Keras](https://medium.com/@hsinhungw/understanding-word-embeddings-with-keras-dfafde0d15a4).
